# 5.3 更新方向怎麼累積？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**梯度突然反向，momentum會立刻完全反向嗎？

momentum把前幾步方向保留一部分，讓反覆一致的方向累積。它可以減少鋸齒，也可能因慣性越過谷底。

**把直覺寫成數學：**v_new=μv+g；w_new=w−ηv_new。這裡用未做偏差修正的基本形式。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
velocity = 0.0
position = 0.0
for gradient in (1.0, 1.0, -1.0):
    velocity = 0.9 * velocity + gradient
    position -= 0.1 * velocity
    print(gradient, velocity, position)

**如何讀結果：**這是給定梯度的數值演算，沒有跑模型訓練。

**只改一件事：**只把μ改成0，比較是否回到gradient descent。
